# Experiment 01 — ICD classification only

This is the classification baseline for Task 2. It uses frozen DINOv3 image features and frozen MedCLIP text features, projects image/text/fused representations into one shared space, and applies one shared multilabel ICD classifier. No metric-learning loss is used here.

The unit of prediction is one study. JSONL rows are grouped by `study_id`, leaf ICD codes become a multi-hot target, and the official MIMIC `train`/`validate`/`test` split is retained.

## 1. Environment

Run `../setup_experiment01.sh` once before building the feature cache. It downloads the official MedCLIP-ViT checkpoint, the DINOv3 ViT-S/16 checkpoint from the Task 2 Kaggle dataset, and a pinned copy of the official Meta DINOv3 loader. Model weights, source, caches, checkpoints, and logs remain inside Task 2.


In [ ]:
from pathlib import Path
import json
import os
import sys

TASK2_ROOT = Path.cwd().resolve()
if TASK2_ROOT.name == 'notebooks':
    TASK2_ROOT = TASK2_ROOT.parent
if not (TASK2_ROOT / 'experiments').is_dir():
    raise RuntimeError('Run this notebook from src/task2 or src/task2/notebooks')

sys.path.insert(0, str(TASK2_ROOT))
os.environ['HF_HOME'] = str(TASK2_ROOT / '.cache' / 'huggingface')
os.environ['TORCH_HOME'] = str(TASK2_ROOT / '.cache' / 'torch')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DIR'] = str(TASK2_ROOT)
os.environ['WANDB_CACHE_DIR'] = str(TASK2_ROOT / '.cache' / 'wandb')

from experiments.experiment01 import (
    ExperimentConfig,
    build_backbone_cache,
    build_label_vocabulary,
    load_study_records,
    run_experiment,
    split_records,
    summarize_records,
)

print(f'Task 2 root: {TASK2_ROOT}')

## 2. Configuration

`min_label_frequency=5` removes leaf codes seen in fewer than five training studies. The value is explicit and saved in the run manifest. Set it to `1` to retain every training code.

In [ ]:
config = ExperimentConfig(
    task2_root=str(TASK2_ROOT),
    mimic_root='/datasets/MIMIC-CXR',
    metadata_path='/datasets/MIMIC-CXR/mimicxr_parsed_ds.jsonl',
    dino_checkpoint_path='pretrained/dinov3-vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth',
    dino_source_dir='vendor/dinov3',
    embedding_dim=512,
    min_label_frequency=5,
    batch_size=16,
    extraction_batch_size=8,
    epochs=20,
    image_loss_weight=0.25,
    text_loss_weight=0.25,
    fused_loss_weight=0.50,
    wandb_project='mimic-icd-task2',
    wandb_run_name='experiment01-classification-notebook',
    wandb_mode='online',
).resolve()
config

## 3. Verify study grouping, split, and label space

The vocabulary is learned from training studies only. Validation/test codes absent from the training vocabulary are ignored, and studies with no remaining target code are excluded.

In [ ]:
records = load_study_records(config.metadata_path, config.mimic_root)
train_records = split_records(records, 'train')
label_codes, label_to_index, label_frequency = build_label_vocabulary(
    train_records, config.min_label_frequency
)

print(json.dumps(summarize_records(records, label_to_index), indent=2))
print(f'Number of predicted leaf ICD codes: {len(label_codes):,}')
print('Most frequent codes:', label_frequency.most_common(10))

## 4. Cache frozen backbone embeddings

This is required only once unless the backbone or source data changes. Set `BUILD_CACHE=1` in the environment or change the flag below. The cache stores every available image embedding and one report embedding per study.

In [ ]:
BUILD_CACHE = os.getenv('BUILD_CACHE', '0') == '1'
cache_path = Path(config.cache_path)

if BUILD_CACHE:
    cache = build_backbone_cache(records, config)
    print(cache['metadata'])
elif cache_path.exists():
    print(f'Using existing cache: {cache_path}')
else:
    print(f'Cache is absent: {cache_path}')
    print('Set BUILD_CACHE=1 before executing this cell or submit the SLURM job.')

## 5. Train and evaluate

The total objective is `0.25 × image BCE + 0.25 × text BCE + 0.50 × fused BCE`. The best checkpoint is selected using validation fused micro average precision. Image, text, and fused predictions are all evaluated. Training configuration, per-epoch losses and metrics, split/cache summaries, and final test metrics are logged to W&B. Set `RUN_TRAINING=1` only in a GPU allocation.


In [ ]:
RUN_TRAINING = os.getenv('RUN_TRAINING', '0') == '1'

if RUN_TRAINING:
    results = run_experiment(config, build_cache=not Path(config.cache_path).exists())
else:
    results = None
    print('Training is disabled. Submit ../slurm/experiment01_classification.sbatch,')
    print('or set RUN_TRAINING=1 inside a GPU allocation.')

In [ ]:
results_path = Path(config.output_dir) / 'results.json'
if results is None and results_path.exists():
    results = json.loads(results_path.read_text())

if results is not None:
    print('Best epoch:', results['best_epoch'])
    print(json.dumps(results['test'], indent=2))

## SLURM launch

From `src/task2`:

```bash
bash setup_experiment01.sh
sbatch slurm/experiment01_classification.sbatch
```

The first job builds the backbone cache. Later runs reuse it. All generated files stay under `src/task2`.